In [1]:
from __future__ import annotations
import ast, os, re, glob, json, difflib, statistics
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field

import pandas as pd

NITE = "{http://nite.sourceforge.net/}"

@dataclass
class Config:
    zayats_tsv: str = "data/raw/zayats/switchboard_corrected_with_silver_reannotation.tsv"
    nxt_dir:    str = "data/raw/nxt/nxt_switchboard_ann/xml"

    # POS="UH" lexicons (split per the paper: insertions are ATYPICAL, fillers TYPICAL)
    filler_words:    set = field(default_factory=lambda: {"uh","um","eh","ah","hm","hum","er"})
    insertion_words: set = field(default_factory=lambda: {"well","like","so"})

    # final surface sweep: tag these as filled_pause even without a POS, where untagged.
    # only unambiguous fillers (never like/well/so, which need the POS gate).
    clear_fillers:   set = field(default_factory=lambda: {"uh","um","eh","ah","hm","er"})

    reparandum_tags: set = field(default_factory=lambda: {"BE","IE","IP","BE_IP","C_IE","C_IP"})
    nonlexical:      set = field(default_factory=lambda:
                        {"--","-","//","[silence]","[noise]","[laughter]","[vocalized-noise]"})

    sil_token:     str   = "<sil>"
    min_agreement: float = 0.90   # don't enrich a conversation below this join quality
    swap_margin:   float = 0.20   # treat as A/B swap if cross-speaker beats self by this much

    # Zayats TSV list-valued columns (parsed with ast.literal_eval)
    list_cols: tuple = ("sentence","ms_sentence","comb_sentence",
                        "names","ms_names","comb_ann","ms_disfl")
    tokens_col: str = "ms_sentence"
    tags_col:   str = "ms_disfl"

In [2]:
# tiny helpers + NXT XML parsing
def safe_float(x):
    try: return float(x)
    except (TypeError, ValueError): return None # "non-aligned" etc. -> None

def pw_key(idstr):
    """uwB2_pw3_a -> ('B',3,'a');  ms1A_pw1 -> ('A',1,'').  None/'None' -> None."""
    if idstr in (None, "None"): return None
    m_spk = re.search(r"(?:uw|ms\d*)([AB])", idstr)
    m_pw  = re.search(r"_pw(\d+)([A-Za-z_]*)$", idstr)
    if not (m_spk and m_pw): return None
    return (m_spk.group(1), int(m_pw.group(1)), m_pw.group(2).lstrip("_"))

FRAG_RE = re.compile(r"^[a-z]+[a-z']*-$") # cut-off word: "h-", "th-"
def is_fragment(t): return bool(FRAG_RE.match(t.lower()))

def _href(ch): return ch.get(f"{NITE}href") or ch.get("href") or ""

def parse_terminals(fp):
    """Ordered stream of dicts. kind in {word, sil}; a word carries orth/pos/phon
       where phon = the MS-State phonword id it points to (NXT's own cross-layer link)."""
    out = []
    for el in ET.parse(fp).getroot():
        tag = el.tag.split('}')[-1]
        if tag == "word":
            phon = None
            for ch in el:
                if ch.tag.split('}')[-1] == "pointer" and ch.get("role") == "phon":
                    m = re.search(r"#id\((.*?)\)", _href(ch)); phon = m.group(1) if m else None
            out.append(dict(kind="word", orth=el.get("orth"), pos=el.get("pos"), phon=phon))
        elif tag == "sil":
            out.append(dict(kind="sil", orth=None, pos=None, phon=None))
    return out

def parse_phonwords(fp):
    """Ordered list of dicts: id / orth (lowercased) = the MS-State transcript."""
    out = []
    for pw in ET.parse(fp).getroot():
        if pw.tag.split('}')[-1] == "phonword":
            out.append(dict(id=pw.get(f"{NITE}id"), orth=(pw.get("orth") or "").lower()))
    return out

def term_phon_path(conv, spk, nxt_dir):
    return (os.path.join(nxt_dir, "terminals", f"{conv}.{spk}.terminals.xml"),
            os.path.join(nxt_dir, "phonwords", f"{conv}.{spk}.phonwords.xml"))

def conv_speaker_ids(nxt_dir):
    ids = []
    for fp in glob.glob(os.path.join(nxt_dir, "terminals", "*.terminals.xml")):
        b = os.path.basename(fp).split("."); ids.append((b[0], b[1]))
    return sorted(set(ids))

In [3]:
def zayats_units(df, conv, spk, tokens_col):
    """Zayats tokens for (conv, spk) grouped into spoken-word UNITS (merging _a/_b
       clitic splits). Returns [(unit_orth_lower, [(row_id, pos), ...])] in order."""
    sub = df[(df["conv"] == conv) & (df["speaker"] == spk)]
    if "turn" in sub: sub = sub.sort_values(["turn", "sent_num"])
    units, cur_key, cur = [], object(), None
    def base(idstr):
        k = pw_key(idstr); return (k[0], k[1]) if k else None
    for rid, r in sub.iterrows():
        toks, names = r[tokens_col], r["ms_names"]
        if not (isinstance(toks, list) and isinstance(names, list)): continue
        for pos, (w, nm) in enumerate(zip(toks, names)):
            b = base(nm)
            if b is not None and b == cur_key:
                cur[0] += w.lower(); cur[1].append((rid, pos))
            else:
                if cur: units.append((cur[0], cur[1]))
                cur = [w.lower(), [(rid, pos)]]; cur_key = b if b is not None else object()
    if cur: units.append((cur[0], cur[1]))
    return units

def _align(phon, units):
    """Core difflib pass. Returns (phonid2row, phon_order, agreement, coverage)
       where phonid2row maps a phonword id -> the LAST Zayats (row,pos) of its unit."""
    a = [p["orth"] for p in phon]
    b = [u for u, _ in units]
    sm = difflib.SequenceMatcher(a=a, b=b, autojunk=False)
    idx2pos, hit, tot = {}, 0, 0
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        if op in ("equal", "replace"):
            for d in range(min(i2 - i1, j2 - j1)):
                idx2pos[i1 + d] = units[j1 + d][1][-1]
                tot += 1; hit += (a[i1 + d] == b[j1 + d])
    order = [p["id"] for p in phon]
    return ({order[i]: rc for i, rc in idx2pos.items()}, order,
            hit / tot if tot else 0.0, len(idx2pos) / len(a) if a else 0.0)

def build_align(df, conv, nxt_spk, cfg, zay_spk=None):
    """Align NXT (conv, nxt_spk) phonwords to Zayats rows of `zay_spk` (default = nxt_spk)."""
    zay_spk = zay_spk or nxt_spk
    fp_t, fp_p = term_phon_path(conv, nxt_spk, cfg.nxt_dir)
    if not (os.path.exists(fp_t) and os.path.exists(fp_p)): return None
    phon  = parse_phonwords(fp_p)
    units = zayats_units(df, conv, zay_spk, cfg.tokens_col)
    if not phon or not units: return None
    phonid2row, order, agree, cov = _align(phon, units)
    return dict(conv=conv, nxt_spk=nxt_spk, zay_spk=zay_spk, agreement=agree, coverage=cov,
                phonid2row=phonid2row, phon_order=order, terminals_path=fp_t, swapped=(zay_spk != nxt_spk))

def build_align_best(df, conv, nxt_spk, cfg):
    """Align to the correct Zayats speaker, auto-detecting an A/B channel swap."""
    other   = "A" if nxt_spk == "B" else "B"
    self_al = build_align(df, conv, nxt_spk, cfg)
    cross   = build_align(df, conv, nxt_spk, cfg, zay_spk=other)
    if self_al is None: return cross            # only the cross-speaker pairing has rows
    if cross is None:   return self_al
    # prefer the swapped pairing only when it clearly wins (avoids spurious swaps)
    return cross if cross["agreement"] > self_al["agreement"] + cfg.swap_margin else self_al

In [4]:
def nxt_tags_for_alignment(df, align, cfg):
    """From one alignment, return:
         typ_flags : {(row_id, pos): 'filled_pause' | 'insertion'}   (POS=UH lexicon)
         sil_after : {(row_id, pos)}  meaning a silent pause occurs AFTER that token
       A tag is only transferred when the mapped Zayats token's surface actually
       equals the NXT word (guards against replace-block mis-tagging)."""
    typ_flags, sil_after = {}, {}
    order_idx = {pid: i for i, pid in enumerate(align["phon_order"])}
    last_phon = None
    for t in parse_terminals(align["terminals_path"]):
        if t["kind"] == "word":
            if t["phon"]: last_phon = t["phon"]
            if t["pos"] != "UH" or not t["phon"]: continue
            orth = (t["orth"] or "").lower()
            cat = ("filled_pause" if orth in cfg.filler_words else
                   "insertion"    if orth in cfg.insertion_words else None)
            if cat is None: continue
            rc = align["phonid2row"].get(t["phon"])
            if not rc: continue
            rid, pos = rc
            toks = df.loc[rid, cfg.tokens_col]
            if pos < len(toks) and toks[pos].lower() == orth:        # surface guard
                typ_flags[rc] = cat
        elif t["kind"] == "sil" and last_phon is not None:
            rc = align["phonid2row"].get(last_phon)
            if rc is None and last_phon in order_idx:                # walk back to nearest mapped
                for i in range(order_idx[last_phon] - 1, -1, -1):
                    rc = align["phonid2row"].get(align["phon_order"][i])
                    if rc: break
            if rc: sil_after[rc] = True
    return typ_flags, sil_after

In [5]:
def token_category(tok, bio_tag, typ_tag, cfg):
    """Single source of truth for a token's disfluency category (or None = fluent)."""
    if tok == cfg.sil_token:           return "silent_pause" # typical
    if bio_tag in cfg.reparandum_tags: return "reparandum" # atypical (Zayats wins)
    if is_fragment(tok):               return "fragment" # atypical
    if typ_tag is not None:            return typ_tag # filled_pause / insertion
    return None # repair 'C' and plain 'O'

_CLITICS = {"'s","'re","'ve","'ll","'d","'m","n't","'t","'em","'all"}
_PUNCT   = {",",".","?","!",";",":","%",")","]","}","''"}
def detok(toks):
    """Join PTB tokens into a readable string (cosmetic only; does not affect counts)."""
    s = ""
    for j, tok in enumerate(toks):
        if j == 0: s = tok
        elif tok.lower() in _CLITICS or tok in _PUNCT or tok.startswith("'"): s += tok
        else: s += " " + tok
    for a, b in {"gon na":"gonna","wan na":"wanna","got ta":"gotta"}.items():
        s = s.replace(a, b)
    return s

def render_pair(tokens, bio, typ, sil_after, cfg):
    """Build fluent & disfluent token lists (+ per-disfluent-token labels) from one row."""
    sil_after = set(sil_after)
    flu, dis, dis_labels, n_disfl = [], [], [], 0
    for i, tok in enumerate(tokens):
        if tok not in cfg.nonlexical:
            cat = token_category(tok, bio[i], typ[i], cfg)
            dis.append(tok); dis_labels.append(cat)
            if cat is None: flu.append(tok)
            else: n_disfl += 1
        if i in sil_after: # silence = disfluent-only token
            dis.append(cfg.sil_token); dis_labels.append("silent_pause"); n_disfl += 1
    return flu, dis, dis_labels, n_disfl

In [6]:
def load_zayats(cfg):
    df = pd.read_csv(cfg.zayats_tsv, sep="\t")
    for c in cfg.list_cols:
        if c in df.columns: df[c] = df[c].apply(ast.literal_eval)
    df["conv"]  = df["file"].str.extract(r"(sw\d+)")
    return df

In [7]:
cfg = Config()
df = load_zayats(cfg)
if "conv" not in df:  df["conv"]  = df["file"].str.extract(r"(sw\d+)")

# align every NXT speaker file -> per-row NXT enrichment (handles swaps)
enrich: dict = {} # row_id -> {"typ": {pos:cat}, "sil": set(pos)}
meta: dict   = {} # row_id -> alignment provenance
for conv, nxt_spk in conv_speaker_ids(cfg.nxt_dir):
    if (df["conv"] == conv).sum() == 0: continue
    al = build_align_best(df, conv, nxt_spk, cfg)
    ok = al["agreement"] >= cfg.min_agreement

    typ_flags, sil_after = nxt_tags_for_alignment(df, al, cfg) if ok else ({}, {})
    for (rid, pos), cat in typ_flags.items():
        enrich.setdefault(rid, {"typ": {}, "sil": set()})["typ"][pos] = cat
    for (rid, pos) in sil_after:
        enrich.setdefault(rid, {"typ": {}, "sil": set()})["sil"].add(pos)
    for rid in df[(df["conv"] == conv) & (df["speaker"] == al["zay_spk"])].index:
        meta[rid] = dict(nxt_speaker=al["nxt_spk"], align_agreement=round(al["agreement"], 3),
                         swapped=al["swapped"], enriched=ok)

# one record per Zayats row
records = []
for rid, r in df.iterrows():
    tokens, bio = r[cfg.tokens_col], r[cfg.tags_col]
    if not (isinstance(tokens, list) and isinstance(bio, list) and len(tokens) == len(bio)):
        continue                                          # skip mis-aligned rows
    e = enrich.get(rid, {"typ": {}, "sil": set()})
    m = meta.get(rid, dict(nxt_speaker=None, align_agreement=None, swapped=None, enriched=False))

    # NXT lexicon tags apply only where Zayats says 'O' (structure wins)
    typ = [e["typ"].get(p) if bio[p] == "O" else None for p in range(len(tokens))]
    # final surface sweep: unambiguous fillers, only where still untagged
    for p, tok in enumerate(tokens):
        if (typ[p] is None and bio[p] == "O" and not is_fragment(tok)
                and tok.lower() in cfg.clear_fillers):
            typ[p] = "filled_pause"
    sil_after = sorted(e["sil"])

    flu, dis, dis_labels, n_disfl = render_pair(tokens, bio, typ, sil_after, cfg)
    records.append({
        # identity / metadata
        "conv": r["conv"], "file": r["file"], "speaker": r["speaker"],
        "turn": int(r["turn"]) if "turn" in r else None,
        "sent_num": int(r["sent_num"]) if "sent_num" in r else None,
        "nxt_speaker": m["nxt_speaker"], "align_agreement": m["align_agreement"],
        "swapped": m["swapped"], "enriched": m["enriched"],

        # 1) raw Zayats tokens   2) Zayats BIO   3) NXT tags + silence positions
        "tokens": tokens, "bio": bio, "typ": typ, "sil_after": sil_after,
        # 4) clean (fluent) string   5) disfluent string (with <sil>)
        "fluent": detok(flu), "disfluent": detok(dis),

        # extras for analysis
        "fluent_tokens": flu, "disfluent_tokens": dis, "disfl_labels": dis_labels,
        "n_tokens_fluent": len(flu), "n_tokens_disfluent": len(dis),
        "n_disfl_tokens": n_disfl,
    })

final_df = pd.DataFrame(records)

In [8]:
final_df.head(50)

,conv,file,speaker,turn,sent_num,nxt_speaker,align_agreement,swapped,enriched,tokens,...,typ,sil_after,fluent,disfluent,fluent_tokens,disfluent_tokens,disfl_labels,n_tokens_fluent,n_tokens_disfluent,n_disfl_tokens
0,sw2005,sw2005.trans,A,1,0,A,0.984,False,True,[okay],...,[None],[],okay,okay,[okay],[okay],[None],1,1,0
1,sw2005,sw2005.trans,A,1,1,A,0.984,False,True,"[uh, first, um, i, need, to, know, uh, h-, how...",...,"[filled_pause, None, filled_pause, None, None,...","[0, 1, 2, 6, 7, 13, 14, 16, 17, 19, 20]",first i need to know how do you feel uh about ...,uh <sil> first <sil> um <sil> i need to know <...,"[first, i, need, to, know, how, do, you, feel,...","[uh, <sil>, first, <sil>, um, <sil>, i, need, ...","[filled_pause, silent_pause, None, silent_paus...",20,38,18
2,sw2005,sw2005.trans,B,2,0,B,0.980,False,True,"[well, of, course, it, 's, you, know, it, 's, ...",...,"[insertion, None, None, None, None, None, None...","[0, 2, 4, 6, 24, 30, 32, 33, 35, 37, 39, 40, 42]",of course you know it's it one of the last few...,well <sil> of course <sil> it's <sil> you know...,"[of, course, you, know, it, 's, it, one, of, t...","[well, <sil>, of, course, <sil>, it, 's, <sil>...","[insertion, silent_pause, None, None, silent_p...",41,60,19
3,sw2005,sw2005.trans,A,3,0,A,0.984,False,True,[yes],...,[None],[],yes,yes,[yes],[yes],[None],1,1,0
4,sw2005,sw2005.trans,A,3,1,A,0.984,False,True,[yeah],...,[None],[],yeah,yeah,[yeah],[yeah],[None],1,1,0
5,sw2005,sw2005.trans,B,4,0,B,0.980,False,True,"[i, 'd, be, very, very, careful, and, uh, you,...",...,"[None, None, None, None, None, None, None, fil...","[6, 7, 9]",i'd be very very careful you know checking the...,i'd be very very careful and <sil> uh <sil> yo...,"[i, 'd, be, very, very, careful, you, know, ch...","[i, 'd, be, very, very, careful, and, <sil>, u...","[None, None, None, None, None, None, reparandu...",11,16,5
6,sw2005,sw2005.trans,B,4,1,B,0.980,False,True,"[uh, our]",...,"[filled_pause, None]","[0, 1]",our,uh <sil> our <sil>,[our],"[uh, <sil>, our, <sil>]","[filled_pause, silent_pause, None, silent_pause]",1,4,3
7,sw2005,sw2005.trans,B,4,2,B,0.980,False,True,"[had, t-, place, my, mother, in, a, nursing, h...",...,"[None, None, None, None, None, None, None, Non...",[1],had place my mother in a nursing home,had t- <sil> place my mother in a nursing home,"[had, place, my, mother, in, a, nursing, home]","[had, t-, <sil>, place, my, mother, in, a, nur...","[None, fragment, silent_pause, None, None, Non...",8,10,2
8,sw2005,sw2005.trans,A,5,0,A,0.984,False,True,[um-hum],...,[None],[],um-hum,um-hum,[um-hum],[um-hum],[None],1,1,0
9,sw2005,sw2005.trans,B,6,0,B,0.980,False,True,"[and, uh, we, were, i, was, fortunate, in, that]",...,"[None, filled_pause, None, None, None, None, N...","[0, 1, 3]",and i was fortunate in that,and <sil> uh <sil> we were <sil> i was fortuna...,"[and, i, was, fortunate, in, that]","[and, <sil>, uh, <sil>, we, were, <sil>, i, wa...","[None, silent_pause, filled_pause, silent_paus...",6,12,6


In [9]:
final_df[['fluent', 'disfluent']].head(50) # sentences for training (?)

,fluent,disfluent
0,okay,okay
1,first i need to know how do you feel uh about ...,uh <sil> first <sil> um <sil> i need to know <...
2,of course you know it's it one of the last few...,well <sil> of course <sil> it's <sil> you know...
3,yes,yes
4,yeah,yeah
5,i'd be very very careful you know checking the...,i'd be very very careful and <sil> uh <sil> yo...
6,our,uh <sil> our <sil>
7,had place my mother in a nursing home,had t- <sil> place my mother in a nursing home
8,um-hum,um-hum
9,and i was fortunate in that,and <sil> uh <sil> we were <sil> i was fortuna...


In [10]:
final_df[['tokens', 'bio', 'typ', 'sil_after']].head(50) # different tags for tokens

,tokens,bio,typ,sil_after
0,[okay],[O],[None],[]
1,"[uh, first, um, i, need, to, know, uh, h-, how...","[O, O, O, O, O, O, O, O, BE_IP, C, O, O, O, BE...","[filled_pause, None, filled_pause, None, None,...","[0, 1, 2, 6, 7, 13, 14, 16, 17, 19, 20]"
2,"[well, of, course, it, 's, you, know, it, 's, ...","[O, O, O, BE, IP, C, C, C, C, O, O, O, O, O, O...","[insertion, None, None, None, None, None, None...","[0, 2, 4, 6, 24, 30, 32, 33, 35, 37, 39, 40, 42]"
3,[yes],[O],[None],[]
4,[yeah],[O],[None],[]
5,"[i, 'd, be, very, very, careful, and, uh, you,...","[O, O, O, O, O, O, BE_IP, O, O, O, O, O, O]","[None, None, None, None, None, None, None, fil...","[6, 7, 9]"
6,"[uh, our]","[O, O]","[filled_pause, None]","[0, 1]"
7,"[had, t-, place, my, mother, in, a, nursing, h...","[O, O, O, O, O, O, O, O, O]","[None, None, None, None, None, None, None, Non...",[1]
8,[um-hum],[O],[None],[]
9,"[and, uh, we, were, i, was, fortunate, in, that]","[O, O, BE, IP, C, C, O, O, O]","[None, filled_pause, None, None, None, None, N...","[0, 1, 3]"


In [12]:
# final_df.to_csv("final_df.csv", index=False)